# Experiment: Case-001 Mitapivat Clinician Conversation Sequence Gate

Objective:
- Validate the ordered clinician conversation sequence before asking about mitapivat.
- Confirm stop conditions prevent jumps to dosing, access, import, travel, or trial screening.

Success criteria:
- six ordered steps are present;
- six allowed clinician response labels are present;
- blocked public content is explicitly listed.


In [ ]:
# Setup: imports and source path
from __future__ import annotations

import json
from pathlib import Path

DATA_PATH = Path(
    "data/regulatory/mitapivat/2026-05-21-mitapivat-clinician-conversation-sequence-gate.json"
)
gate = json.loads(DATA_PATH.read_text(encoding="utf-8"))
gate["gate_label"]

## Gate Contract

This gate keeps the conversation ordered. The mitapivat review-readiness
question is allowed only after scope, core labels, missing records, and the
seven-domain readiness gate are closed.


In [ ]:
required_steps = [
    "open_scope",
    "validate_core_case_labels",
    "close_missing_records",
    "confirm_seven_domain_readiness",
    "ask_mitapivat_review_readiness",
    "route_after_answer",
]

allowed_labels = {
    "review_not_ready_missing_records",
    "not_clinically_relevant_now",
    "review_with_hematologist_only",
    "refer_to_trial_or_access_owner_after_clinician_review",
    "safety_review_required_before_any_discussion",
    "lane_paused_until_pediatric_results_or_recruitment",
}

blocked_content = set(gate["blocked_public_content"])
required_blocked_content = {
    "raw_records",
    "exact_private_dates",
    "screenshots",
    "doctor_messages",
    "owner_replies",
    "contact_details",
    "patient_specific_treatment_instructions",
    "dose_or_dosing_requests",
    "access_import_or_travel_instructions",
    "trial_screening_instruction",
}

sequence = gate["conversation_sequence"]
assert gate["gate_label"] == "mitapivat_clinician_conversation_sequence_ready"
assert gate["upstream_gate"] == "mitapivat_seven_domain_owner_action_map_ready"
assert [item["step_label"] for item in sequence] == required_steps
assert set(gate["allowed_clinician_response_labels"]) == allowed_labels
assert blocked_content == required_blocked_content
assert len(gate["source_checks"]) >= 6

for item in sequence:
    assert item["question_intent"]
    assert item["allowed_public_output"]
    assert item["stop_if"]

decision = {
    "gate_label": gate["gate_label"],
    "steps_checked": len(sequence),
    "allowed_labels_checked": len(gate["allowed_clinician_response_labels"]),
    "public_action": "ask_mitapivat_review_only_after_prerequisites_pass",
}
decision

## Results

- The sequence starts with label validation and missing-record routing.
- The mitapivat review-readiness question is step five, not step one.
- Any missing, corrected, or release-blocked prerequisite stops the lane.


## Next steps

- Use the general doctor handoff before this mitapivat sequence.
- Record only public-safe labels after clinician review.
- Route any answer through the May 18 action ladder.
